<a href="https://colab.research.google.com/github/naokityokoyama/fake_news_hdc/blob/main/torchhd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch-hd torchmetrics binhd -q

In [ ]:
!pip install gensim

In [ ]:
import os
import sys
import torch
import torchhd
import itertools
from torchhd.datasets.isolet import ISOLET
from binhd.datasets import BaseDataset
from binhd.classifiers import BinHD
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
import warnings
# Ignora todos os avisos do tipo Warning
warnings.filterwarnings('ignore')

In [ ]:
#build dataset
def build_dataset(isot, covid, fever):
  if isot:
    df = pd.read_csv('/content/drive/MyDrive/uff/isot.csv')
    # Convert target column to categorical and then to codes to ensure integer labels starting from 0

    return df
  elif covid:
    df = pd.read_csv('/content/drive/MyDrive/uff/covid.csv')
    # Convert target column to categorical and then to codes to ensure integer labels starting from 0

    return df
  elif fever:
    df = pd.read_csv('/content/drive/MyDrive/uff/fever.csv')
    # Convert target column to categorical and then to codes to ensure integer labels starting from 0

    return df

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using {} device".format(device))

df = build_dataset(isot=False, covid=True, fever=False)
X_T, X_t, y_T, y_t = train_test_split(df['frase'], df['target'], test_size=0.30, random_state = 42)

Using cpu device


In [ ]:
df.shape

(5975, 3)

In [ ]:
len(X_T), len(X_t)

(4182, 1793)

TFIDF BATCH

In [ ]:
classifiers = [
    "Vanilla",
    "AdaptHD",
    "OnlineHD",
    "NeuralHD",
    "DistHD",
]




In [ ]:
params = {
    "Vanilla": {},
    "AdaptHD": {
        "epochs": 2,
    },
    "OnlineHD": {
        "epochs": 2,
    },
    "NeuralHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "DistHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "CompHD": {},
    "SparseHD": {
        "epochs": 2,
    },
    "QuantHD": {
        "epochs": 2,
    },
    "LeHDC": {
        "epochs": 2,
    },
    "IntRVFL": {},
}

In [ ]:
batch_size = 1000
num_samples = df.shape[0]
tfidf = TfidfVectorizer(max_features=100)
#tfidf = CountVectorizer(max_features=1000)

In [ ]:
def model_torch_hd(model_name, dimension):

  DIMENSIONS = dimension  # number of hypervector dimension
  num_classes = 2

  lst_acuracia = []
  lst_f1 = []
  lst_precision = []
  lst_recall = []
  lst_start = []
  lst_end = []
  lst_cm = []
  lst_roc = []


  for i in tqdm(range(0, num_samples, batch_size)):
  #start time
    start = time.perf_counter()
    x_ =  tfidf.fit_transform(df['frase']).toarray().astype('float32')[i:i+batch_size]
    y_ = df['target'].values[i:i+batch_size]

    X_train, X_test, y_train, y_test = train_test_split(x_, y_, test_size=0.30, random_state = 42)

    train_dataset = BaseDataset(X_train, y_train)
    test_dataset = BaseDataset(X_test, y_test)

    train_ld = torch.utils.data.DataLoader(train_dataset)
    test_ld = torch.utils.data.DataLoader(test_dataset)

    num_features = x_.shape[1] # Update num_features for each batch

    model_cls = getattr(torchhd.classifiers, model_name)
    model: torchhd.classifiers.Classifier = model_cls(
          num_features, DIMENSIONS, num_classes, device=device, **params[model_name]
      )

    X_test_tmp = torch.tensor(X_test).to(device)

    # Redirect stderr to devnull to suppress tqdm output from internal fit
    old_stderr = sys.stderr
    with open(os.devnull, 'w') as f:
        sys.stderr = f
        model.fit(train_ld)
    sys.stderr = old_stderr

    accuracy = model.accuracy(test_ld)

    # --- Start of fix: Predict in batches ---
    all_y_pred = []
    with torch.no_grad(): # Disable gradient calculations for inference
        for X_batch, _ in test_ld: # Iterate through batches from test_ld
            X_batch = X_batch.to(device)
            batch_y_pred = model.predict(X_batch)
            all_y_pred.append(batch_y_pred.cpu().numpy())

    y_pred = np.concatenate(all_y_pred)
    # --- End of fix ---

    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=[0,1])
    roc = roc_auc_score(y_test, y_pred)

    #end
    end = time.perf_counter()

    lst_acuracia.append(accuracy)
    lst_f1.append(f1)
    lst_precision.append(precision)
    lst_recall.append(recall)
    lst_cm.append(cm)
    lst_roc.append(roc)
    lst_start.append(start)
    lst_end.append(end)

  print ('model', model_cls.__name__)
  print(f"Testing accuracy of {(np.mean(lst_acuracia) * 100):.3f}%")
  print(f"Testing F1 of {(np.mean(lst_f1) * 100):.3f}%")
  print(f"Testing RECALL of {(np.mean(lst_recall) * 100):.3f}%")
  print(f"Testing PRECISION of {(np.mean(lst_precision) * 100):.3f}%")
  print(f"Confusion Matrix of ", np.mean(lst_cm, axis=0))
  print(f"ROC of ", np.nanmean(lst_roc))
  print(f"Size train", len(X_T))
  print(f"Size test", len(X_t))
  print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
  print(f'DIMENSION', DIMENSIONS)
  print ('\n')
  print ('------------------------------------------------')


In [ ]:
model_torch_hd('NeuralHD', dimension=1000)

  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
dimension = [1000, 3000]
for classe, dim  in  itertools.product(classifiers, dimension):

  model_torch_hd(classe, dim)

  0%|          | 0/6 [00:00<?, ?it/s]

model Vanilla
Testing accuracy of 77.574%
Testing F1 of 39.522%
Testing RECALL of 57.331%
Testing PRECISION of 35.048%
Confusion Matrix of  [[182.83333333  45.        ]
 [ 22.          49.        ]]
ROC of  0.708808908270405
Size train 4182
Size test 1793
Time total: 8.788545 s
DIMENSION 1000


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

model Vanilla
Testing accuracy of 83.574%
Testing F1 of 47.071%
Testing RECALL of 59.992%
Testing PRECISION of 42.020%
Confusion Matrix of  [[198.33333333  29.5       ]
 [ 19.66666667  51.33333333]]
ROC of  0.7639699740972146
Size train 4182
Size test 1793
Time total: 7.825970 s
DIMENSION 3000


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

...

model NeuralHD
Testing accuracy of 86.525%
Testing F1 of 49.769%
Testing RECALL of 58.397%
Testing PRECISION of 44.850%
Confusion Matrix of  [[203.5         24.33333333]
 [ 16.          55.        ]]
ROC of  0.7664311358918376
Size train 4182
Size test 1793
Time total: 11.249456 s
DIMENSION 3000


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

model DistHD
Testing accuracy of 86.624%
Testing F1 of 45.338%
Testing RECALL of 48.729%
Testing PRECISION of 43.707%
Confusion Matrix of  [[203.83333333  24.        ]
 [ 16.          55.        ]]
ROC of  0.6908605223537264
Size train 4182
Size test 1793
Time total: 14.031922 s
DIMENSION 1000


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

model DistHD
Testing accuracy of 85.513%
Testing F1 of 42.127%
Testing RECALL of 44.038%
Testing PRECISION of 42.297%
Confusion Matrix of  [[203.16666667  24.66666667]
 [ 18.66666667  52.33333333]]
ROC of  0.6550579820300103
Size train 4182
Size test 1793
Time total: 14.021157 s
DIMENSION 3000


------------------------------------------------


W2V

In [ ]:
import re
import gensim.downloader as api

In [ ]:
#download corpus w2v 300d
wv = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
#encoding
def tokenize(text: str):
    return re.findall(r"\b\w+\b", text.lower())

def sentence_embedding(text: str) -> np.ndarray:
    tokens = tokenize(text)
    vecs = [wv[t] for t in tokens if t in wv]
    if not vecs:
        return np.zeros(wv.vector_size, dtype=np.float32)
    #calculando a media (Mean Pooling)
    return np.mean(vecs, axis=0).astype(np.float32)

def texts_to_matrix(texts):
    return np.vstack([sentence_embedding(t) for t in tqdm(texts)])

In [ ]:
embedding = texts_to_matrix(df['text'])

  0%|          | 0/5975 [00:00<?, ?it/s]

In [ ]:
embedding

array([[ 0.02616882,  0.06614176,  0.1023763 , ..., -0.00487264,
         0.03845596, -0.00407918],
       [ 0.05819702, -0.00334167, -0.00300598, ..., -0.05059814,
         0.08750153,  0.00598145],
       [ 0.04760742,  0.01046499, -0.01570638, ..., -0.12516277,
         0.07784017,  0.04211426],
       ...,
       [ 0.05721436, -0.03068848,  0.0003418 , ..., -0.10285644,
         0.05880585, -0.01481934],
       [ 0.0131073 ,  0.01785362,  0.06735992, ..., -0.03743744,
        -0.00933552,  0.00279236],
       [ 0.07038371,  0.02547236,  0.02522659, ...,  0.02706273,
         0.02938589, -0.07040501]], dtype=float32)

In [ ]:
y = df['target'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(embedding, y, test_size=0.30, random_state = 42)

In [ ]:
classifiers = [
    #"Vanilla",
    "AdaptHD",
    #"OnlineHD",
    #"NeuralHD",
    #"DistHD",



]



DIMENSIONS = 3000  # number of hypervector dimensions
BATCH_SIZE = 12  # for GPUs with enough memory we can process multiple

num_features = 300
num_classes = 2

In [ ]:
train_dataset = BaseDataset(X_train, y_train)
test_dataset = BaseDataset(X_test, y_test)

train_ld = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE)
test_ld = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
params = {
    "Vanilla": {},
    "AdaptHD": {
        "epochs": 2,
    },
    "OnlineHD": {
        "epochs": 2,
    },
    "NeuralHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "DistHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "CompHD": {},
    "SparseHD": {
        "epochs": 2,
    },
    "QuantHD": {
        "epochs": 2,
    },
    "LeHDC": {
        "epochs": 2,
    },
    "IntRVFL": {},
}

In [ ]:
import os
import sys

lst_start = []
lst_end = []
for classifier in classifiers:
    #start time
    start = time.perf_counter()
    print(classifier)

    #set model
    model_cls = getattr(torchhd.classifiers, classifier)
    model: torchhd.classifiers.Classifier = model_cls(
        num_features, DIMENSIONS, num_classes, device=device, **params[classifier]
    )

    # Redirect stderr to devnull to suppress tqdm output from internal fit
    old_stderr = sys.stderr
    with open(os.devnull, 'w') as f:
        sys.stderr = f
        model.fit(train_ld)
    sys.stderr = old_stderr

    accuracy = model.accuracy(test_ld)

    # --- Start of fix: Predict in batches ---
    all_y_pred = []
    with torch.no_grad(): # Disable gradient calculations for inference
        for X_batch, _ in test_ld: # Iterate through batches from test_ld
            X_batch = X_batch.to(device)
            batch_y_pred = model.predict(X_batch)
            all_y_pred.append(batch_y_pred.cpu().numpy())

    y_pred = np.concatenate(all_y_pred)
    # --- End of fix ---


    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred)

    #end
    end = time.perf_counter()

    lst_start.append(start)
    lst_end.append(end)
    print(f"Testing accuracy of {(accuracy * 100):.3f}%")
    print(f"Testing F1 of {(f1 * 100):.3f}%")
    print(f"Testing RECALL of {(recall * 100):.3f}%")
    print(f"Testing PRECISION of {(precision * 100):.3f}%")
    print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
    print (f'Confusion Matrix', cm)
    print(f"ROC of  {(roc * 100):.3f}%")
    print(f"Size train", len(X_train))
    print(f"Size test", len(X_test))
    print(f'DIMENSION', DIMENSIONS)


AdaptHD
Testing accuracy of 81.651%
Testing F1 of 56.653%
Testing RECALL of 48.753%
Testing PRECISION of 67.610%
Time total: 3.328422 s
Confusion Matrix [[1249  103]
 [ 226  215]]
ROC of  70.567%
Size train 4182
Size test 1793
DIMENSION 3000


BOW

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
bow = CountVectorizer(max_features=700)

In [ ]:
X = bow.fit_transform(df['frase']).toarray().astype('float32')

y = df['target'].values

In [ ]:
X.shape

(5975, 700)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state = 20, stratify=y)

In [ ]:
classifiers = [
    #"Vanilla",
    "AdaptHD",
    #"OnlineHD",
    #"NeuralHD",
    #"DistHD",



]



DIMENSIONS = 1000  # number of hypervector dimensions
BATCH_SIZE = 128  # for GPUs with enough memory we can process multiple

num_features = 700
num_classes = 2

In [ ]:
train_dataset = BaseDataset(X_train, y_train)
test_dataset = BaseDataset(X_test, y_test)

train_ld = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE)
test_ld = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
params = {
    "Vanilla": {},
    "AdaptHD": {
        "epochs": 2,
    },
    "OnlineHD": {
        "epochs": 2,
    },
    "NeuralHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "DistHD": {
        "epochs": 2,
        "regen_freq": 5,
    },
    "CompHD": {},
    "SparseHD": {
        "epochs": 2,
    },
    "QuantHD": {
        "epochs": 2,
    },
    "LeHDC": {
        "epochs": 2,
    },
    "IntRVFL": {},
}

In [ ]:
import os
import sys

lst_start = []
lst_end = []
for classifier in classifiers:
    #start time
    start = time.perf_counter()
    print(classifier)

    #set model
    model_cls = getattr(torchhd.classifiers, classifier)
    model: torchhd.classifiers.Classifier = model_cls(
        num_features, DIMENSIONS, num_classes, device=device, **params[classifier]
    )

    # X_test_tmp = torch.tensor(X_test).to(device) # Original line - removed

    # Redirect stderr to devnull to suppress tqdm output from internal fit
    old_stderr = sys.stderr
    with open(os.devnull, 'w') as f:
        sys.stderr = f
        model.fit(train_ld)
    sys.stderr = old_stderr

    accuracy = model.accuracy(test_ld)

    # --- Start of fix: Predict in batches ---
    all_y_pred = []
    with torch.no_grad(): # Disable gradient calculations for inference
        for X_batch, _ in test_ld: # Iterate through batches from test_ld
            X_batch = X_batch.to(device)
            batch_y_pred = model.predict(X_batch)
            all_y_pred.append(batch_y_pred.cpu().numpy())

    y_pred = np.concatenate(all_y_pred)
    # --- End of fix ---


    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=[0,1])
    roc = roc_auc_score(y_test, y_pred)
    #end
    end = time.perf_counter()
    lst_start.append(start)
    lst_end.append(end)

    print(f"Testing accuracy of {(accuracy * 100):.3f}%")
    print(f"Testing F1 of {(f1 * 100):.3f}%")
    print(f"Testing RECALL of {(recall * 100):.3f}%")
    print(f"Testing PRECISION of {(precision * 100):.3f}%")
    print (f'Confusion Matrix', cm)
    print(f"ROC of  {(roc * 100):.3f}%")
    print(f"Size train", len(X_T))
    print(f"Size test", len(X_t))
    print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
    print(f'DIMENSION', DIMENSIONS)


AdaptHD
Testing accuracy of 53.597%
Testing F1 of 46.667%
Testing RECALL of 84.065%
Testing PRECISION of 32.298%
Confusion Matrix [[597 763]
 [ 69 364]]
ROC of  63.981%
Size train 4182
Size test 1793
Time total: 72.980359 s
DIMENSION 1000


In [ ]:
tempo = 26.06
(tempo*12000 )/500

625.44